In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.random_projection import SparseRandomProjection
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import csr_matrix, hstack
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

In [4]:
DATA_PATH = "data/train_lang.csv"

TEXT_COL = "sentence"
RATING_COL = "label"
LANG_COL = "lang"

df = pd.read_csv(DATA_PATH)
df = df[[TEXT_COL, RATING_COL, LANG_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[RATING_COL] = df[RATING_COL].astype(int)

df["strat"] = df[LANG_COL].astype(str) + "_" + df[RATING_COL].astype(str)

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["strat"]
)

y_train = train_df[RATING_COL].values.astype(float)
y_val = val_df[RATING_COL].values.astype(float)

print(train_df.shape, val_df.shape)

(226800, 4) (25200, 4)


In [5]:
word_vec = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=2,
    max_features=30_000,
    sublinear_tf=True,
    lowercase=True
)

char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=2,
    max_features=30_000,
    sublinear_tf=True,
    lowercase=True
)

Xw_tr = word_vec.fit_transform(train_df[TEXT_COL])
Xw_va = word_vec.transform(val_df[TEXT_COL])

Xc_tr = char_vec.fit_transform(train_df[TEXT_COL])
Xc_va = char_vec.transform(val_df[TEXT_COL])

X_tr_sparse = hstack([Xw_tr, Xc_tr]).tocsr()
X_va_sparse = hstack([Xw_va, Xc_va]).tocsr()

print(X_tr_sparse.shape)

: 

In [ ]:
ridge = Ridge(alpha=10.0)
ridge.fit(X_tr_sparse, y_train)

pred_sparse = np.clip(ridge.predict(X_va_sparse), 1, 5)
print("Sparse TF-IDF Ridge MAE:", mean_absolute_error(y_val, pred_sparse))

NameError: name 'Ridge' is not defined

In [ ]:
rp = SparseRandomProjection(
    n_components=2048,
    density="auto",
    random_state=42
)

X_tr_jl = rp.fit_transform(X_tr_sparse)
X_va_jl = rp.transform(X_va_sparse)

print(X_tr_jl.shape)

In [ ]:
ridge_jl = Ridge(alpha=10.0)
ridge_jl.fit(X_tr_jl, y_train)

pred_jl = np.clip(ridge_jl.predict(X_va_jl), 1, 5)
print("JL embedding Ridge MAE:", mean_absolute_error(y_val, pred_jl))